# Tutorial 07 — Retrieval-augmented generation and evidence-based answers

[Lecture notes](../lectures/07_rag.md) · [Exercise sheet](../exercises/07_rag.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Index an authored campus document collection, compute retrieval metrics, and return cited evidence or abstain.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Build a transparent text index

All campus rules are fictional and authored for this lab. The retrieval corpus is available evidence, not a set of held-out question-answer pairs.

Before running: predict the output or the invariant being checked.


In [2]:
import re
from collections import Counter
docs={"D1":"The library opens at eight on weekdays.","D2":"The computing lab closes at eighteen on weekdays.","D3":"Borrowed books must be returned within seven days.","D4":"The study room contains a projector and a whiteboard."}
def tokens(s): return re.findall(r"\w+",s.lower())
vocab=sorted(set(t for d in docs.values() for t in tokens(d))); vid={t:i for i,t in enumerate(vocab)}
def bow(s):
    x=np.zeros(len(vocab))
    for t,c in Counter(tokens(s)).items():
        if t in vid: x[vid[t]]=c
    return x
counts=np.stack([bow(d) for d in docs.values()])
idf=np.log((1+len(docs))/(1+(counts>0).sum(0)))+1
X=counts*idf; X/=np.maximum(np.linalg.norm(X,axis=1,keepdims=True),1e-12)
docids=list(docs)
def retrieve(q,k=2):
    v=bow(q)*idf; v/=max(np.linalg.norm(v),1e-12)
    scores=X@v; order=np.argsort(-scores,kind="stable")[:k]
    return [(docids[i],float(scores[i])) for i in order]
print(retrieve("When does the computing lab close?"))


[('D2', 0.6195345281036431), ('D1', 0.11435668050043521)]


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Evaluate retrieval separately

Each authored query has one relevant document. We report hit@k, not general multi-document recall.

Before running: predict the output or the invariant being checked.


In [3]:
queries=[("When does the library open?","D1"),("When does the lab close?","D2"),("How long may books be borrowed?","D3"),("What equipment is in the study room?","D4")]
ranks=[]
for q,gold in queries:
    order=[d for d,_ in retrieve(q,len(docs))]
    ranks.append(order.index(gold)+1)
print("ranks",ranks,"MRR",np.mean(1/np.array(ranks)),"hit@1",np.mean(np.array(ranks)<=1))
assert len(ranks)==len(queries)


ranks [1, 1, 1, 1] MRR 1.0 hit@1 1.0


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Cited extraction and missing evidence

This returns a source sentence, not a generated answer. The threshold is illustrative; lexical overlap can retrieve irrelevant evidence and must be calibrated on development queries.

Before running: predict the output or the invariant being checked.


In [4]:
def extract(q,threshold=.2):
    docid,score=retrieve(q,1)[0]
    if score<threshold: return {"answer":None,"sources":[],"score":score}
    return {"answer":docs[docid],"sources":[docid],"score":score}
for q in ["library opens", "astronaut volcano", "Does the library offer medical treatment?"]:
    print(q,extract(q))
assert extract("astronaut volcano")["answer"] is None
# Inspect the medical-treatment query: overlap is not evidence that it is answerable.


library opens {'answer': 'The library opens at eight on weekdays.', 'sources': ['D1'], 'score': 0.6159132378921828}
astronaut volcano {'answer': None, 'sources': [], 'score': 0.0}
Does the library offer medical treatment? {'answer': 'The library opens at eight on weekdays.', 'sources': ['D1'], 'score': 0.5166719821867511}


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Can a correct answer have an incorrect citation?
2. Why must document IDs survive chunking?
3. What does an oracle-context experiment isolate?


## Extension and submission

Pass retrieved evidence into `extensions/hf_text.py` as a prompt and score generated claims manually. Compare with oracle and shuffled evidence; the core lab uses extraction, not generated answers.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
